<a href="https://colab.research.google.com/github/mhovhannisyan06/handf/blob/main/baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HanDF baseline: pix2tex vs Gemini
Reads my labelled formula crops, runs both models, and scores them.

**Before running:**
1. In Drive, `handf_data/formulas/` contains crops like `page_001_f01.png`.
2. Your `labels` Google Sheet (columns `image`, `latex`) is downloaded as **CSV** and uploaded to `handf_data/labels.csv` (File → Download → CSV).
3. Gemini API key from https://aistudio.google.com/apikey saved in Colab: 🔑 **Secrets** (left sidebar) → name `GEMINI_API_KEY`, turn on *Notebook access*.

Run the cells top to bottom (Shift+Enter).

**Runtime → Change runtime type → T4 GPU** (if available), then **Runtime → Run all**. Gemini answers are read from `gemini_cache.json`, so no new Gemini requests are made.

In [1]:
# 1) Install the two models' libraries (takes 1-2 minutes)
!pip install -q pix2tex google-genai

In [2]:
# 2) Connect Drive and load labels
from google.colab import drive
drive.mount('/content/drive')

import os, re, time
import pandas as pd

BASE = '/content/drive/MyDrive/handf_data'
CROPS = os.path.join(BASE, 'formulas')

labels = pd.read_csv(os.path.join(BASE, 'labels.csv'))
labels = labels.dropna(subset=['image', 'latex'])
labels = labels[labels['image'].apply(lambda f: os.path.exists(os.path.join(CROPS, f)))]
print(len(labels), 'labelled formulas with an image file')
labels.head()

Mounted at /content/drive
52 labelled formulas with an image file


,image,latex
0,page_001_f01.png,50a - a^2 = 576
1,page_001_f02.png,a^2 - 50a + 576 = 0
2,page_001_f03.png,D = 2500 - 2304
3,page_001_f04.png,\frac{50 \pm 14}{2}
4,page_001_f05.png,d^2 = R^2 - 2Rr =


In [3]:
# 3) Scoring: make small formatting differences not count
def normalize(s):
    s = str(s).strip().strip('$')
    s = re.sub(r'^```(latex)?|```$', '', s.strip()).strip()
    for cmd in [r'\\left', r'\\right', r'\\displaystyle', r'\\,', r'\\;', r'\\!', r'\\quad', r'\\ ']:
        s = re.sub(cmd, '', s)
    s = re.sub(r'\s+', '', s)                       # remove all spaces
    s = s.replace('\\limits', '').replace('\\dfrac', '\\frac').replace('\\tfrac', '\\frac')
    for _ in range(3):
        s = re.sub(r'\{([A-Za-z0-9])\}', r'\1', s)      # x^{2} -> x^2
        s = re.sub(r'\{(\\[A-Za-z]+)\}', r'\1', s)   # {\alpha} -> \alpha
    return s

def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]

def score(pred, truth):
    p, t = normalize(pred), normalize(truth)
    exact = int(p == t)
    cer = edit_distance(p, t) / max(len(t), 1)   # share of characters wrong
    return exact, min(cer, 1.0)

print(normalize('x^{2} + \\left( a \\right)'))  # should print: x^2+(a)

x^2+(a)


In [4]:
# 4) pix2tex (no training, as it comes)
import torch, time
from PIL import Image
print('GPU available:', torch.cuda.is_available())

print('Loading pix2tex model (first time downloads it)...')
t = time.time()
from pix2tex.cli import LatexOCR
p2t = LatexOCR()
print('Model loaded in', round(time.time() - t), 'seconds')

pix_preds = []
for i, f in enumerate(labels['image'], 1):
    try:
        img = Image.open(os.path.join(CROPS, f)).convert('RGB')
        pred = p2t(img)
    except Exception as e:
        print('pix2tex error on', f, e)
        pred = ''
    pix_preds.append(pred)
    print(f'{i}/{len(labels)}  {f}  ->  {pred[:60]}')
labels['pix2tex'] = pix_preds
print('pix2tex done')

GPU available: True
Loading pix2tex model (first time downloads it)...


/usr/local/lib/python3.13/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.24). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
/usr/local/lib/python3.13/dist-packages/pydantic/_internal/_serializers.py:47: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `dict[str, any]` - serialized value may not be as expected [field_name='noise_params', input_value=UniformParams(noise_type=... 0.058823529411764705)]), input_type=UniformParams])
  item_v = handler(item, index)
/usr/local/lib/python3.13/dist-packages/pydantic/main.py:520: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `dict[str, any]` - serialized value may not be as expected [field_name='noise_params', input_value=UniformParams(noise_type=... 0.0588235294117647

download weights v0.0.1 to path /usr/local/lib/python3.13/dist-packages/pix2tex/model/checkpoints


weights.pth: 100%|██████████| 97.4M/97.4M [00:00<00:00, 126Mb/s]
image_resizer.pth: 100%|██████████| 18.5M/18.5M [00:00<00:00, 32.6Mb/s]


Model loaded in 16 seconds
1/52  page_001_f01.png  ->  \bar{f}\overline{{{\bf\Omega}\alpha\Omega}\stackrel{\leftarr
2/52  page_001_f02.png  ->  \Omega\Delta_{\mathrm{l-}}S\delta_{\mathrm{s}}\mathrm{s}\mat
3/52  page_001_f03.png  ->  \mathbb{S}{\stackrel{+}{\hookrightarrow}}\,2.5\,\mathbb{\oti
4/52  page_001_f04.png  ->  \frac{\zeta_{\textstyle\sum\div}^{}}{\cfrac{\mathbb{Q}_{\mat
5/52  page_001_f05.png  ->  \begin{array}{r}{{\dot{\bigcup_{-+}^{1}{\frac{1}{\bigcap{t}}
6/52  page_002_f01.png  ->  {\mathcal{L}}[={\bar{C}}(0]
7/52  page_002_f02.png  ->  \frac{|\hat{y_{1}},\ \vartheta_{-}|\hat{y_{-}}|\hat{z_{-}}|\
8/52  page_002_f03.png  ->  \scriptstyle{3+d o w=N}
9/52  page_003_f01.png  ->  L\iiint_{0}^{n}\!d z\vert\,{\frac{1\vert\langle\langle\langl
10/52  page_003_f02.png  ->  \frac{\left|\widetilde{{\mathcal J}^{1}}^{-}(\tilde{\Sigma}^
11/52  page_003_f03.png  ->  \frac{{\sqrt{\textstyle\sum_{\mathrm{i}}^{-}}}{\textstyle\le
12/52  page_003_f04.png  ->  \left|\sum_{j}\frac{1}{\iiint_{0}

## Experiment: does cleaning the background help pix2tex?

In [5]:
# 4b) Clean the images, then pix2tex again
# What clean() does, step by step:
#   1. take the RED colour channel: blue ink looks dark there, light-blue grid looks light
#   2. blur it very strongly -> the ink disappears, only the paper background is left
#   3. subtract: (background - image) -> only what is clearly darker than the paper (= ink) stays
#   4. Otsu threshold -> black/white; then delete tiny specks (noise)
# Result: black handwriting on a clean white background, like the printed formulas pix2tex learned from.
# The cleaned images are also saved to Drive: handf_data/formulas_clean/ (open them and look!)
import cv2, numpy as np

CLEAN_DIR = os.path.join(BASE, 'formulas_clean')
os.makedirs(CLEAN_DIR, exist_ok=True)

def clean(path):
    im = cv2.imread(path)
    r = cv2.GaussianBlur(im[:, :, 2], (3, 3), 0)          # 1. red channel
    bg = cv2.medianBlur(r, 31)                             # 2. paper background
    diff = cv2.subtract(bg, r)                             # 3. ink = darker than paper
    _, ink = cv2.threshold(diff, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)   # 4. black/white
    n, lab, st, _ = cv2.connectedComponentsWithStats(ink, 8)
    keep = np.zeros_like(ink)
    for i in range(1, n):
        if st[i, cv2.CC_STAT_AREA] >= 12:                  #    drop tiny specks
            keep[lab == i] = 255
    out = cv2.copyMakeBorder(255 - keep, 15, 15, 15, 15, cv2.BORDER_CONSTANT, value=255)
    return Image.fromarray(out).convert('RGB')

clean_preds = []
for i, f in enumerate(labels['image'], 1):
    img = clean(os.path.join(CROPS, f))
    img.save(os.path.join(CLEAN_DIR, f))
    try:
        pred = p2t(img)
    except Exception as e:
        print('error on', f, e); pred = ''
    clean_preds.append(pred)
    print(f'{i}/{len(labels)}  {f}  ->  {pred[:60]}')
labels['pix2tex_clean'] = clean_preds
print('pix2tex on cleaned images done')

1/52  page_001_f01.png  ->  S\triangleright_{\mathbf{a}}\cdot\Leftrightarrow\ {\stackrel
2/52  page_001_f02.png  ->  \mathrm{a}^{2}\setminus\cdots\mathrm{sosyrit}\lor
3/52  page_001_f03.png  ->  \mathbf{3}:2S\ 6\mathbf{c}-2J{\mathrm{ov~}}\mathbf{-}
4/52  page_001_f04.png  ->  \frac{5}{\frac{z}{\rightarrow}}\frac{z^{*}\{\mathbb{V}}{\pm}
5/52  page_001_f05.png  ->  \v_{\parallel}^{1},\ \P^{\Lambda}\cdot\ \|\Lambda\,t^{\Xi},
6/52  page_002_f01.png  ->  3(m)={\frac{1}{2}}\mathop{|}_{0}n\qquad\qquad
7/52  page_002_f02.png  ->  \begin{array}{r l}{\left\{\S_{a}{\frac{9}{2\pi_{\circ}}}}&{{
8/52  page_002_f03.png  ->  \scriptstyle{S_{n}(\mathbf{a}-\mathbf{g})}
9/52  page_003_f01.png  ->  16\hat{\mathrm{l}}\hat{\mathrm{l}}_{\pm}^{+}\hat{\Xi}_{\pm}^
10/52  page_003_f02.png  ->  \exists{\overset{\sim}{\to}}{\overset{\frown}{\supset}}|\equ
11/52  page_003_f03.png  ->  \frac{-{\sqrt{\frac{2}{\pi}}}t}{\sqrt{\frac{2}{\sqrt{\zeta}}
12/52  page_003_f04.png  ->  \frac{4}{\displaystyle\int\!\!\int\!\!\int

In [6]:
# 5) Gemini — slow and careful, so we stay inside the free limits
# Answers are saved to Drive (gemini_cache.json) after every formula.
# If it stops (limit reached), just run this cell again later: it continues where it stopped.
import json, time
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
MODEL = 'gemini-3.5-flash-lite'   # each model has its own daily limit; if this one is used up, try 'gemini-3.1-flash-lite'
PROMPT = ('Transcribe the handwritten math formula in this image into LaTeX. '
          'Output only the LaTeX code: no $ signs, no explanation.')
CACHE = os.path.join(BASE, 'gemini_cache.json')
cache = json.load(open(CACHE)) if os.path.exists(CACHE) else {}
print(len(cache), 'answers already saved from before')

def ask(img):
    for attempt in range(5):
        try:
            r = client.models.generate_content(model=MODEL, contents=[img, PROMPT])
            return r.text.strip()
        except Exception as e:
            msg = str(e)
            if 'PerDay' in msg or 'per day' in msg:
                raise RuntimeError('DAILY LIMIT reached for ' + MODEL + '. Try another MODEL or run again tomorrow.')
            wait = 20 * (attempt + 1)
            print('   busy/limit, waiting', wait, 's ...')
            time.sleep(wait)
    return None

for i, f in enumerate(labels['image'], 1):
    if f in cache:
        continue
    img = Image.open(os.path.join(CROPS, f)).convert('RGB')
    try:
        ans = ask(img)
    except RuntimeError as e:
        print(e)
        break
    if ans is not None:
        cache[f] = ans
        json.dump(cache, open(CACHE, 'w'), ensure_ascii=False, indent=1)
    print(f'{i}/{len(labels)}  {f}  ->  {(ans or "FAILED")[:60]}')
    time.sleep(13)   # free tier: max ~5 requests per minute

labels['gemini'] = [cache.get(f, '') for f in labels['image']]
print('Gemini answered', len(cache), 'of', len(labels))

52 answers already saved from before
Gemini answered 52 of 52


In [7]:
# 6) Results — all models compared on the SAME formulas (those Gemini answered)
models = ['pix2tex', 'pix2tex_clean', 'gemini']
for m in models:
    s = labels.apply(lambda r: score(r[m], r['latex']), axis=1)
    labels[m + '_exact'] = [x[0] for x in s]
    labels[m + '_cer'] = [x[1] for x in s]

both = labels[labels['gemini'] != '']
summary = pd.DataFrame([{
    'model': m,
    'exact match %': round(100 * both[m + '_exact'].mean(), 1),
    'avg char error %': round(100 * both[m + '_cer'].mean(), 1),
    'n formulas': len(both)} for m in models])
labels.to_csv(os.path.join(BASE, 'baseline_results.csv'), index=False)
print('Saved all predictions to handf_data/baseline_results.csv')
summary

Saved all predictions to handf_data/baseline_results.csv


,model,exact match %,avg char error %,n formulas
0,pix2tex,0.0,99.9,52
1,pix2tex_clean,0.0,98.1,52
2,gemini,59.6,10.2,52


In [8]:
# 7) Look at mistakes: what does each model get wrong?
for m in ['pix2tex_clean', 'gemini']:
    print(f'\n===== {m}: 5 worst =====')
    for _, r in labels.sort_values(m + '_cer', ascending=False).head(5).iterrows():
        print(r['image'])
        print('  correct:', r['latex'])
        print('  model:  ', r[m])


===== pix2tex_clean: 5 worst =====
page_001_f01.png
  correct: 50a - a^2 = 576
  model:   S\triangleright_{\mathbf{a}}\cdot\Leftrightarrow\ {\stackrel{1}{\sim}}-{\underline{{\zeta}}}\,{\overline{{\chi}}}\,\zeta
page_001_f02.png
  correct: a^2 - 50a + 576 = 0
  model:   \mathrm{a}^{2}\setminus\cdots\mathrm{sosyrit}\lor
page_001_f03.png
  correct: D = 2500 - 2304
  model:   \mathbf{3}:2S\ 6\mathbf{c}-2J{\mathrm{ov~}}\mathbf{-}
page_001_f04.png
  correct: \frac{50 \pm 14}{2}
  model:   \frac{5}{\frac{z}{\rightarrow}}\frac{z^{*}\{\mathbb{V}}{\pm}
page_001_f05.png
  correct: d^2 = R^2 - 2Rr =
  model:   \v_{\parallel}^{1},\ \P^{\Lambda}\cdot\ \|\Lambda\,t^{\Xi},

===== gemini: 5 worst =====
page_003_f01.png
  correct: |\overrightarrow{BA}| = 3
  model:   | bA | = 3
page_003_f04.png
  correct: |\overrightarrow{BL}| = \sqrt{20} = 2\sqrt{5}
  model:   {15}^{2} = \sqrt{20} = 2\sqrt{5}
page_004_f10.png
  correct: k_0 = \left[\frac{1}{4\varepsilon} - \frac{3}{4}\right] + 1
  model:   k_0 = \lflo

In [9]:
!pip install -q "optimum[onnxruntime]"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [10]:
# NEW MODEL: Pix2Text-MFR (trained on printed AND handwritten formulas)
from transformers import TrOCRProcessor
from optimum.onnxruntime import ORTModelForVision2Seq

processor = TrOCRProcessor.from_pretrained('breezedeus/pix2text-mfr')
mfr = ORTModelForVision2Seq.from_pretrained('breezedeus/pix2text-mfr', use_cache=False)

def read_mfr(img):
    pv = processor(images=[img], return_tensors='pt').pixel_values
    ids = mfr.generate(pv, max_new_tokens=256)
    return processor.batch_decode(ids, skip_special_tokens=True)[0]

orig, cleaned = [], []
for i, f in enumerate(labels['image'], 1):
    path = os.path.join(CROPS, f)
    try:
        a = read_mfr(Image.open(path).convert('RGB'))   # original photo
        b = read_mfr(clean(path))                        # cleaned (uses clean() from cell 4b)
    except Exception as e:
        print('error on', f, e); a = b = ''
    orig.append(a); cleaned.append(b)
    print(f'{i}/{len(labels)}  {f}\n    original -> {a[:60]}\n    cleaned  -> {b[:60]}')
labels['mfr'] = orig
labels['mfr_clean'] = cleaned

# Results for ALL models, on the same 52 formulas
rows = []
for m in ['pix2tex', 'pix2tex_clean', 'mfr', 'mfr_clean', 'gemini']:
    s = labels.apply(lambda r: score(r[m], r['latex']), axis=1)
    labels[m + '_exact'] = [x[0] for x in s]
    labels[m + '_cer'] = [x[1] for x in s]
    rows.append({'model': m,
                 'exact match %': round(100 * labels[m + '_exact'].mean(), 1),
                 'avg char error %': round(100 * labels[m + '_cer'].mean(), 1)})
labels.to_csv(os.path.join(BASE, 'baseline_results.csv'), index=False)
pd.DataFrame(rows)

ValueError: Backend should be defined in the BACKENDS_MAPPING. Offending backend: tensorflow_text

In [14]:
!pip install -q "transformers>=4.37,<5" "optimum[onnxruntime]"

In [15]:
import transformers
print(transformers.__version__)

5.22.1


In [13]:
# NEW MODEL: Pix2Text-MFR — self-contained cell (run after Restart session)
from google.colab import drive
drive.mount('/content/drive')
import os, re, cv2, numpy as np, pandas as pd
from PIL import Image

BASE  = '/content/drive/MyDrive/handf_data'
CROPS = os.path.join(BASE, 'formulas')
labels = pd.read_csv(os.path.join(BASE, 'baseline_results.csv'), keep_default_na=False)
print(len(labels), 'formulas loaded, columns:', [c for c in labels.columns if not c.endswith(('_exact', '_cer'))])

# --- scoring (same rules as before) ---
def normalize(s):
    s = str(s).strip().strip('$')
    s = re.sub(r'^```(latex)?|```$', '', s.strip()).strip()
    for cmd in [r'\\left', r'\\right', r'\\displaystyle', r'\\,', r'\\;', r'\\!', r'\\quad', r'\\ ']:
        s = re.sub(cmd, '', s)
    s = s.replace('\\limits', '').replace('\\dfrac', '\\frac').replace('\\tfrac', '\\frac')
    s = re.sub(r'\s+', '', s)
    for _ in range(3):
        s = re.sub(r'\{([A-Za-z0-9])\}', r'\1', s)
        s = re.sub(r'\{(\\[A-Za-z]+)\}', r'\1', s)
    return s

def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]

def score(pred, truth):
    p, t = normalize(pred), normalize(truth)
    return int(p == t), min(edit_distance(p, t) / max(len(t), 1), 1.0)

# --- cleaning (same as cell 4b) ---
def clean(path):
    im = cv2.imread(path)
    r = cv2.GaussianBlur(im[:, :, 2], (3, 3), 0)
    bg = cv2.medianBlur(r, 31)
    diff = cv2.subtract(bg, r)
    _, ink = cv2.threshold(diff, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, lab, st, _ = cv2.connectedComponentsWithStats(ink, 8)
    keep = np.zeros_like(ink)
    for i in range(1, n):
        if st[i, cv2.CC_STAT_AREA] >= 12:
            keep[lab == i] = 255
    out = cv2.copyMakeBorder(255 - keep, 15, 15, 15, 15, cv2.BORDER_CONSTANT, value=255)
    return Image.fromarray(out).convert('RGB')

# --- the new model ---
from transformers import TrOCRProcessor
from optimum.onnxruntime import ORTModelForVision2Seq
processor = TrOCRProcessor.from_pretrained('breezedeus/pix2text-mfr')
mfr = ORTModelForVision2Seq.from_pretrained('breezedeus/pix2text-mfr', use_cache=False)

def read_mfr(img):
    pv = processor(images=[img], return_tensors='pt').pixel_values
    ids = mfr.generate(pv, max_new_tokens=256)
    return processor.batch_decode(ids, skip_special_tokens=True)[0]

orig, cleaned = [], []
for i, f in enumerate(labels['image'], 1):
    path = os.path.join(CROPS, f)
    try:
        a = read_mfr(Image.open(path).convert('RGB'))
        b = read_mfr(clean(path))
    except Exception as e:
        print('error on', f, e); a = b = ''
    orig.append(a); cleaned.append(b)
    print(f'{i}/{len(labels)}  {f}\n    original -> {a[:60]}\n    cleaned  -> {b[:60]}')
labels['mfr'] = orig
labels['mfr_clean'] = cleaned

# --- results for all models ---
models = [m for m in ['pix2tex', 'pix2tex_clean', 'mfr', 'mfr_clean', 'gemini'] if m in labels.columns]
rows = []
for m in models:
    s = labels.apply(lambda r: score(r[m], r['latex']), axis=1)
    labels[m + '_exact'] = [x[0] for x in s]
    labels[m + '_cer'] = [x[1] for x in s]
    rows.append({'model': m,
                 'exact match %': round(100 * labels[m + '_exact'].mean(), 1),
                 'avg char error %': round(100 * labels[m + '_cer'].mean(), 1)})
labels.to_csv(os.path.join(BASE, 'baseline_results.csv'), index=False)
pd.DataFrame(rows)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
52 formulas loaded, columns: ['image', 'latex', 'pix2tex', 'pix2tex_clean', 'gemini']


ValueError: Backend should be defined in the BACKENDS_MAPPING. Offending backend: tensorflow_text

In [16]:
# NEW MODEL: Pix2Text-MFR via ONNX Runtime (no transformers needed) — self-contained cell
from google.colab import drive
drive.mount('/content/drive')
import os, re, json, cv2, numpy as np, pandas as pd
from PIL import Image
import onnxruntime as ort
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer

BASE  = '/content/drive/MyDrive/handf_data'
CROPS = os.path.join(BASE, 'formulas')
labels = pd.read_csv(os.path.join(BASE, 'baseline_results.csv'), keep_default_na=False)
labels = labels[[c for c in labels.columns if not c.endswith(('_exact', '_cer'))]]
print(len(labels), 'formulas loaded, columns:', list(labels.columns))

# --- scoring (same rules as before) ---
def normalize(s):
    s = str(s).strip().strip('$')
    s = re.sub(r'^```(latex)?|```$', '', s.strip()).strip()
    for cmd in [r'\\left', r'\\right', r'\\displaystyle', r'\\,', r'\;', r'\\!', r'\\quad', r'\\ ']:
        s = re.sub(cmd, '', s)
    s = s.replace('\\limits', '').replace('\\dfrac', '\\frac').replace('\\tfrac', '\\frac')
    s = re.sub(r'\s+', '', s)
    for _ in range(3):
        s = re.sub(r'\{([A-Za-z0-9])\}', r'\1', s)
        s = re.sub(r'\{(\\[A-Za-z]+)\}', r'\1', s)
    return s

def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]

def score(pred, truth):
    p, t = normalize(pred), normalize(truth)
    return int(p == t), min(edit_distance(p, t) / max(len(t), 1), 1.0)

# --- cleaning (same as cell 4b) ---
def clean(path):
    im = cv2.imread(path)
    r = cv2.GaussianBlur(im[:, :, 2], (3, 3), 0)
    bg = cv2.medianBlur(r, 31)
    diff = cv2.subtract(bg, r)
    _, ink = cv2.threshold(diff, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, lab, st, _ = cv2.connectedComponentsWithStats(ink, 8)
    keep = np.zeros_like(ink)
    for i in range(1, n):
        if st[i, cv2.CC_STAT_AREA] >= 12:
            keep[lab == i] = 255
    out = cv2.copyMakeBorder(255 - keep, 15, 15, 15, 15, cv2.BORDER_CONSTANT, value=255)
    return Image.fromarray(out).convert('RGB')

# --- load the new model (ONNX files: an encoder that looks at the image, a decoder that writes LaTeX) ---
REPO = 'breezedeus/pix2text-mfr'
get = lambda name: hf_hub_download(REPO, name)
cfg  = json.load(open(get('config.json')))
pre  = json.load(open(get('preprocessor_config.json')))
try:
    gen = json.load(open(get('generation_config.json')))
except Exception:
    gen = {}
tok  = Tokenizer.from_file(get('tokenizer.json'))
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if 'CUDAExecutionProvider' in ort.get_available_providers() else ['CPUExecutionProvider']
enc = ort.InferenceSession(get('encoder_model.onnx'), providers=providers)
dec = ort.InferenceSession(get('decoder_model.onnx'), providers=providers)
dec_inputs = [i.name for i in dec.get_inputs()]
print('decoder inputs:', dec_inputs)

start_id = gen.get('decoder_start_token_id', cfg.get('decoder_start_token_id', cfg.get('decoder', {}).get('decoder_start_token_id', 2)))
eos_id   = gen.get('eos_token_id', cfg.get('eos_token_id', cfg.get('decoder', {}).get('eos_token_id', 2)))
if isinstance(eos_id, list): eos_id = eos_id[0]
size = pre.get('size', {'height': 384, 'width': 384})
H, W = (size['height'], size['width']) if isinstance(size, dict) else (size, size)
mean = np.array(pre.get('image_mean', [0.5, 0.5, 0.5]), dtype=np.float32)
std  = np.array(pre.get('image_std',  [0.5, 0.5, 0.5]), dtype=np.float32)
scale = pre.get('rescale_factor', 1 / 255)

def read_mfr(img, max_len=256):
    x = np.asarray(img.convert('RGB').resize((W, H), Image.BICUBIC), dtype=np.float32) * scale
    x = ((x - mean) / std).transpose(2, 0, 1)[None].astype(np.float32)
    hidden = enc.run(None, {enc.get_inputs()[0].name: x})[0]
    ids = [start_id]
    for _ in range(max_len):
        feed = {}
        for name in dec_inputs:
            if 'input_ids' in name:
                feed[name] = np.array([ids], dtype=np.int64)
            elif 'encoder_hidden_states' in name:
                feed[name] = hidden
            elif 'attention_mask' in name:
                feed[name] = np.ones(hidden.shape[:2], dtype=np.int64)
        logits = dec.run(None, feed)[0]
        nxt = int(logits[0, -1].argmax())
        if nxt == eos_id:
            break
        ids.append(nxt)
    return tok.decode(ids[1:], skip_special_tokens=True)

orig, cleaned = [], []
for i, f in enumerate(labels['image'], 1):
    path = os.path.join(CROPS, f)
    try:
        a = read_mfr(Image.open(path))
        b = read_mfr(clean(path))
    except Exception as e:
        print('error on', f, e); a = b = ''
    orig.append(a); cleaned.append(b)
    print(f'{i}/{len(labels)}  {f}\n    original -> {a[:60]}\n    cleaned  -> {b[:60]}')
labels['mfr'] = orig
labels['mfr_clean'] = cleaned

# --- results for all models ---
models = [m for m in ['pix2tex', 'pix2tex_clean', 'mfr', 'mfr_clean', 'gemini'] if m in labels.columns]
rows = []
for m in models:
    s = labels.apply(lambda r: score(r[m], r['latex']), axis=1)
    labels[m + '_exact'] = [x[0] for x in s]
    labels[m + '_cer'] = [x[1] for x in s]
    rows.append({'model': m,
                 'exact match %': round(100 * labels[m + '_exact'].mean(), 1),
                 'avg char error %': round(100 * labels[m + '_cer'].mean(), 1)})
labels.to_csv(os.path.join(BASE, 'baseline_results.csv'), index=False)
pd.DataFrame(rows)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
52 formulas loaded, columns: ['image', 'latex', 'pix2tex', 'pix2tex_clean', 'gemini']


config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/450 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/39.2k [00:00<?, ?B/s]

encoder_model.onnx: reconstructing file:   0%|          |  0.00B / 87.5MB            

encoder_model.onnx: downloading bytes:           |  0.00B            

decoder_model.onnx: reconstructing file:   0%|          |  0.00B / 30.1MB            

decoder_model.onnx: downloading bytes:           |  0.00B            

decoder inputs: ['input_ids', 'encoder_hidden_states']
1/52  page_001_f01.png
    original -> 5 0 a - a ^ { 2 } = 5 + 6
    cleaned  -> 5 0 \pi - \pi ^ { 2 } - 5 7 6
2/52  page_001_f02.png
    original -> a ^ { 2 } - 5 0 a + 5 + 6 = 0
    cleaned  -> a ^ { 2 } \cdot 5 \circ a \pm 5 7 6 \cdot
3/52  page_001_f03.png
    original -> \textcircled { 3 } = 2 5 0 0 - 2 3 0 ^ { 4 }
    cleaned  -> 3 5 \div 2 5 0 0 - 2 3 0 4
4/52  page_001_f04.png
    original -> \frac { 5 0 \pm 1 4 } { 2 }
    cleaned  -> \frac { 5 0 \pm 1 4 } { 2 }
5/52  page_001_f05.png
    original -> d ^ { 2 } - R ^ { 2 } - 2 R t =
    cleaned  -> d ^ { 2 } - R ^ { 2 } - 2 R _ { 2 } =
6/52  page_002_f01.png
    original -> x = 6 0 ^ { \circ }
    cleaned  -> \dot { \omega _ { 2 } } ^ { \downarrow } \omega ^ { \circ }
7/52  page_002_f02.png
    original -> \frac { 5 \cdot 9 } { 2 } = \frac { \sqrt { 3 } } { 2 }
    cleaned  -> \frac { 5 } { 2 } . . \frac { \sqrt { 2 } } { 2 } .
8/52  page_002_f03.png
    original -> 3 + 1 0

,model,exact match %,avg char error %
0,pix2tex,0.0,99.9
1,pix2tex_clean,0.0,98.1
2,mfr,15.4,40.7
3,mfr_clean,5.8,53.9
4,gemini,59.6,10.2
